# 10 — Anomaly Injection
**Spacecraft Telemetry Anomaly Detection | Phase 2**

---
**Goal:** Inject realistic synthetic anomalies into the clean normal telemetry to create a labelled dataset for model evaluation.

> In real spacecraft operations, anomalies are rare and rarely labelled in advance.
> We simulate them here so we can measure model performance objectively.

**Three anomaly types injected:**
- **Point** — single extreme spike or drop
- **Contextual** — wrong value for the time/orbital context
- **Collective** — a sequence of readings with abnormal drift or oscillation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
                     'axes.titlesize':11, 'font.size':10})
os.makedirs('plots_v2', exist_ok=True)
os.makedirs('data', exist_ok=True)

PALETTE = ['#1f77b4','#2ca02c','#d62728','#9467bd','#8c564b',
           '#e377c2','#7f7f7f','#bcbd22','#17becf','#ff7f0e']

# Load clean baseline telemetry
tel = pd.read_csv('telemetry_train.csv')
tel['timestamp'] = pd.to_datetime(tel['timestamp'])
tel = tel.sort_values(['parameter','timestamp']).reset_index(drop=True)

# Initialise anomaly label columns
tel['is_anomaly']   = 0       # 0 = normal, 1 = anomaly
tel['anomaly_type'] = 'none'  # 'none' | 'point' | 'contextual' | 'collective'

print('Clean telemetry loaded:', tel.shape)
print('Parameters:', tel['parameter'].nunique())

### 10.1 Compute Per-Parameter Statistics
We need each parameter's mean and std to place anomalies at realistic but extreme offsets.

In [ ]:
# Compute normal distribution stats per parameter — used to calculate anomaly magnitudes
param_stats = (tel.groupby('parameter')['value']
               .agg(['mean','std','min','max'])
               .rename(columns={'mean':'p_mean','std':'p_std',
                                'min':'p_min','max':'p_max'}))

print('Per-parameter stats computed:')
display(param_stats.round(3))

### 10.2 Type 1 — Point Anomalies

A **single reading** is replaced with an extreme value (4–6 standard deviations from the mean).

> *Physical scenario:* A sensor glitch, cosmic ray bit-flip, or momentary hardware fault
> produces one completely spurious reading before recovering to normal.

In [ ]:
# One parameter selected from each subsystem for point anomalies
POINT_PARAMS = {
    'Power':      'BATT_VOLTAGE_1',
    'Thermal':    'OBC_TEMP',
    'ADCS':       'GYRO_X',
    'Comms':      'RF_SIGNAL_STRENGTH',
    'OBC':        'CPU_USAGE',
    'Propulsion': 'TANK_PRESSURE',
}

N_POINT_PER_PARAM = 8  # 8 point anomalies per parameter × 6 params = 48 total

for subsys, param in POINT_PARAMS.items():
    mask   = tel['parameter'] == param
    idx    = tel[mask].index.tolist()
    chosen = np.random.choice(idx, size=N_POINT_PER_PARAM, replace=False)

    mu  = param_stats.loc[param, 'p_mean']
    sig = param_stats.loc[param, 'p_std']

    for i in chosen:
        # Randomly spike upward or downward by 4–6 std devs
        direction = np.random.choice([-1, 1])
        magnitude = np.random.uniform(4, 6)
        tel.at[i, 'value']        = mu + direction * magnitude * sig
        tel.at[i, 'is_anomaly']   = 1
        tel.at[i, 'anomaly_type'] = 'point'

    print(f'  [{subsys:12s}] {param:30s} — {N_POINT_PER_PARAM} point anomalies injected')

n_point = (tel['anomaly_type'] == 'point').sum()
print(f'\nTotal point anomalies: {n_point}')

### 10.3 Type 2 — Contextual Anomalies

A value that is **physically wrong for its time context** — the number itself may look
plausible in isolation, but given the hour/orbital phase it makes no sense.

> *Physical scenario:* `SOLAR_POWER_TOTAL` reporting near-zero watts during the orbital
> dayside pass — should be ~100 W. Possible cause: deployable solar panel stuck folded.

In [ ]:
# Contextual anomalies: wrong value given the hour-of-day context
# Hours 6-18 = dayside (sunlit) | Hours 0-5 & 19-23 = nightside (eclipse)
CONTEXTUAL_CASES = [
    # (parameter, anomaly_hours, forced_value_fn)
    # Solar power near-zero during dayside
    ('SOLAR_POWER_TOTAL',
     list(range(7, 17)),
     lambda mu, sig: np.random.uniform(0.5, 3.0)),
    # Battery voltage suspiciously HIGH during eclipse (should be discharging)
    ('BATT_VOLTAGE_2',
     list(range(0, 5)) + list(range(20, 24)),
     lambda mu, sig: mu + np.random.uniform(2.5, 4.0) * sig),
    # RF signal very strong at low elevation hours (spacecraft over horizon)
    ('RF_SIGNAL_STRENGTH',
     list(range(0, 24)),
     lambda mu, sig: mu + np.random.uniform(3.0, 4.5) * sig),
]

N_CONTEXTUAL_PER_CASE = 10

tel['hour'] = tel['timestamp'].dt.hour  # temporary column

for param, anom_hours, val_fn in CONTEXTUAL_CASES:
    mu  = param_stats.loc[param, 'p_mean']
    sig = param_stats.loc[param, 'p_std']

    mask = (tel['parameter'] == param) & (tel['hour'].isin(anom_hours))
    idx  = tel[mask & (tel['is_anomaly'] == 0)].index.tolist()

    if len(idx) >= N_CONTEXTUAL_PER_CASE:
        chosen = np.random.choice(idx, size=N_CONTEXTUAL_PER_CASE, replace=False)
        for i in chosen:
            tel.at[i, 'value']        = val_fn(mu, sig)
            tel.at[i, 'is_anomaly']   = 1
            tel.at[i, 'anomaly_type'] = 'contextual'
        print(f'  {param:30s} — {N_CONTEXTUAL_PER_CASE} contextual anomalies injected (hours={anom_hours[:3]}...)')
    else:
        print(f'  {param:30s} — insufficient rows in target hours ({len(idx)} available)')

tel.drop(columns=['hour'], inplace=True)  # remove temp column

n_contextual = (tel['anomaly_type'] == 'contextual').sum()
print(f'\nTotal contextual anomalies: {n_contextual}')

### 10.4 Type 3 — Collective Anomalies

A **contiguous window of readings** that together represent an abnormal pattern —
no single point looks extreme, but the sequence as a whole deviates from expected behaviour.

> *Physical scenarios:*
> - Slow monotonic drift (e.g., tank pressure steadily declining → propellant leak)
> - Sudden step-change followed by sustained offset (e.g., gyro bias shift after manoeuvre)
> - Unusual oscillation (e.g., thermal oscillation from a stuck radiator valve)

In [ ]:
# Collective anomaly patterns
COLLECTIVE_CASES = [
    # (parameter, pattern, window_size, n_windows)
    ('TANK_PRESSURE',    'drift_down', 25, 3),   # slow pressure loss
    ('GYRO_Y',           'step_shift', 20, 3),   # attitude control bias shift
    ('MEMORY_USAGE',     'drift_up',   30, 2),   # memory leak pattern
    ('RADIATOR_TEMP',  'oscillate',  20, 3),   # thermal oscillation
    ('BATT_VOLTAGE_1',   'drift_down', 20, 2),   # battery degradation pattern
]

def inject_collective(tel, param, pattern, window_size, n_windows):
    """Inject a collective anomaly pattern into a contiguous window."""
    mask    = (tel['parameter'] == param) & (tel['is_anomaly'] == 0)
    indices = tel[mask].index.tolist()
    mu      = param_stats.loc[param, 'p_mean']
    sig     = param_stats.loc[param, 'p_std']
    count   = 0

    for _ in range(n_windows):
        if len(indices) < window_size + 10:
            break
        # Pick a random starting position well within the available indices
        start_pos = np.random.randint(5, len(indices) - window_size - 5)
        window    = indices[start_pos: start_pos + window_size]

        if pattern == 'drift_down':
            # Gradual monotonic decline over the window
            drift = np.linspace(0, -3.0 * sig, window_size)
            for j, idx in enumerate(window):
                if tel.at[idx, 'is_anomaly'] == 0:
                    tel.at[idx, 'value']        += drift[j]
                    tel.at[idx, 'is_anomaly']   = 1
                    tel.at[idx, 'anomaly_type'] = 'collective'
                    count += 1

        elif pattern == 'drift_up':
            drift = np.linspace(0, +3.0 * sig, window_size)
            for j, idx in enumerate(window):
                if tel.at[idx, 'is_anomaly'] == 0:
                    tel.at[idx, 'value']        += drift[j]
                    tel.at[idx, 'is_anomaly']   = 1
                    tel.at[idx, 'anomaly_type'] = 'collective'
                    count += 1

        elif pattern == 'step_shift':
            # Sudden persistent offset — the signal jumps and stays there
            offset = np.random.choice([-1, 1]) * np.random.uniform(2.5, 3.5) * sig
            for idx in window:
                if tel.at[idx, 'is_anomaly'] == 0:
                    tel.at[idx, 'value']        += offset
                    tel.at[idx, 'is_anomaly']   = 1
                    tel.at[idx, 'anomaly_type'] = 'collective'
                    count += 1

        elif pattern == 'oscillate':
            # Sinusoidal oscillation superimposed on normal signal
            freq  = np.random.uniform(0.5, 1.5)  # cycles per window
            amp   = np.random.uniform(2.0, 3.0) * sig
            phase = np.linspace(0, 2 * np.pi * freq, window_size)
            for j, idx in enumerate(window):
                if tel.at[idx, 'is_anomaly'] == 0:
                    tel.at[idx, 'value']        += amp * np.sin(phase[j])
                    tel.at[idx, 'is_anomaly']   = 1
                    tel.at[idx, 'anomaly_type'] = 'collective'
                    count += 1

        # Remove injected indices from pool to avoid overlap
        injected_set = set(window)
        indices = [i for i in indices if i not in injected_set]

    return count


for param, pattern, win, n_win in COLLECTIVE_CASES:
    cnt = inject_collective(tel, param, pattern, win, n_win)
    print(f'  {param:30s} | {pattern:12s} | window={win} x {n_win} windows → {cnt} points injected')

n_collective = (tel['anomaly_type'] == 'collective').sum()
print(f'\nTotal collective anomalies: {n_collective}')

### 10.5 Injection Summary

In [ ]:
total_anomalies = tel['is_anomaly'].sum()
anomaly_rate    = total_anomalies / len(tel) * 100

summary = tel.groupby('anomaly_type').size().reset_index(name='count')
summary['percentage'] = (summary['count'] / len(tel) * 100).round(2)

print(f'Total rows          : {len(tel):,}')
print(f'Total anomalies     : {total_anomalies}')
print(f'Anomaly rate        : {anomaly_rate:.2f}%')
print()
display(summary)

# Per-parameter anomaly count
print('\nPer-parameter anomaly breakdown:')
per_param = (tel[tel['is_anomaly']==1]
             .groupby(['parameter','anomaly_type'])
             .size()
             .reset_index(name='count')
             .sort_values(['anomaly_type','parameter']))
display(per_param)

### 10.6 Visualise — Before vs After Injection

In [ ]:
# Load the original clean dataset for comparison
tel_clean = pd.read_csv('telemetry_train.csv')
tel_clean['timestamp'] = pd.to_datetime(tel_clean['timestamp'])

SHOW_PARAMS = ['BATT_VOLTAGE_1','TANK_PRESSURE','GYRO_Y',
               'SOLAR_POWER_TOTAL','MEMORY_USAGE','RADIATOR_TEMP']

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle('Anomaly Injection — Before (blue) vs After (anomalies in red)',
             fontsize=13, fontweight='bold')

anom_colors = {'point':'#d62728', 'contextual':'#ff7f0e', 'collective':'#9467bd'}

for ax, param in zip(axes.flatten(), SHOW_PARAMS):
    clean_sub = tel_clean[tel_clean['parameter']==param].sort_values('timestamp')
    anom_sub  = tel[tel['parameter']==param].sort_values('timestamp')

    # Plot clean baseline
    ax.plot(clean_sub['timestamp'], clean_sub['value'],
            color='#1f77b4', lw=0.8, alpha=0.5, label='Normal')

    # Overlay anomalous points coloured by type
    for atype, color in anom_colors.items():
        sub = anom_sub[anom_sub['anomaly_type'] == atype]
        if len(sub) > 0:
            ax.scatter(sub['timestamp'], sub['value'],
                       color=color, s=30, zorder=5, label=atype)

    ax.set_title(param, fontweight='bold', fontsize=9)
    ax.set_ylabel('Value', fontsize=8)
    ax.legend(fontsize=7, loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('plots_v2/10_anomaly_injection.png', dpi=150, bbox_inches='tight')
plt.show()

# RESULT: Red dots (point) = clear spikes isolated from the normal blue line
# Orange dots (contextual) = within plausible range but at the wrong time
# Purple sequence (collective) = a sustained drift or oscillation segment

### 10.7 Anomaly Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Injected Anomaly Distribution', fontsize=12, fontweight='bold')

# Pie chart — anomaly type breakdown
type_counts = tel[tel['is_anomaly']==1]['anomaly_type'].value_counts()
axes[0].pie(type_counts.values,
            labels=type_counts.index,
            autopct='%1.1f%%',
            colors=['#d62728','#ff7f0e','#9467bd'],
            startangle=90, pctdistance=0.8)
axes[0].set_title('By Anomaly Type', fontweight='bold')

# Bar chart — anomalies per parameter
per_param_total = (tel[tel['is_anomaly']==1]
                   .groupby('parameter').size()
                   .sort_values(ascending=False))
per_param_total.plot(kind='bar', ax=axes[1],
                     color='#1f77b4', edgecolor='white', alpha=0.85)
axes[1].set_title('Anomaly Count per Parameter', fontweight='bold')
axes[1].set_xlabel('Parameter')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig('plots_v2/10b_anomaly_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save labelled dataset — this is the input for all Phase 2 models
tel.to_csv('data/telemetry_with_anomalies.csv', index=False)
print('Saved: data/telemetry_with_anomalies.csv')
print('Shape:', tel.shape)
print('Columns:', list(tel.columns))
print(f'\nNormal rows   : {(tel["is_anomaly"]==0).sum():,}')
print(f'Anomaly rows  : {(tel["is_anomaly"]==1).sum():,}  ({anomaly_rate:.2f}% of dataset)')